# Trial-2: Complete Training Pipeline - Google Colab Version

**IMPORTANT SETUP STEPS:**
1. Runtime → Change runtime type → GPU (T4 or A100)
2. Upload `Refine-Chunked/` folder to your Google Drive
3. Upload Trial-2 architecture files to Google Drive
4. Run cells in order

**Expected training time:** ~3-4 hours for 150 epochs on T4 GPU

**Key Improvements Over Trial-1:**
- U-Net skip connections
- Multi-head self-attention  
- Multi-scale dilated convolutions
- Scenario conditioning
- Enhanced data augmentation (5 techniques)
- 4-stage curriculum learning
- Perceptual loss

## Step 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("\nDrive mounted successfully!")
print("Available files:")
!ls "/content/drive/MyDrive/October Thesis Dataset/"

Mounted at /content/drive

Drive mounted successfully!
Available files:
checkpoints  Refine-Chunked  results


## Step 2: Check GPU Availability

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not enabled! Go to Runtime → Change runtime type → GPU")

PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB
GPU Memory: 42.47 GB


## Step 3: Install Additional Dependencies (if needed)

In [3]:
# Install any missing packages
!pip install -q scipy scikit-learn tqdm
print("✅ Dependencies installed")

✅ Dependencies installed


## Step 4: Configuration - TRIAL-2 Settings

In [4]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
import numpy as np
import json
import glob
from pathlib import Path
import time
import warnings
from scipy.interpolate import interp1d
import scipy.signal as signal
warnings.filterwarnings('ignore')

# Configuration
CONFIG = {
    # Paths (Google Drive)
    'chunk_dir': '/content/drive/MyDrive/October Thesis Dataset/Refine-Chunked',
    'split_file': '/content/drive/MyDrive/October Thesis Dataset/Refine-Chunked/train_val_test_split.json',
    'checkpoint_dir': '/content/drive/MyDrive/October Thesis Dataset/checkpoints',
    'results_dir': '/content/drive/MyDrive/October Thesis Dataset/results',

    # Data - REDUCED FOR MEMORY
    'batch_size': 4,  # Reduced from 16 to 4 due to attention memory
    'num_workers': 2,
    'pin_memory': True,

    # Model - Trial-2 Enhanced
    'signal_length': 64000,
    'base_channels': 64,
    'num_scenarios': 3,  # rest, exercise, recovery
    'use_attention': True,
    'use_dilated': True,
    'use_skip_connections': True,

    # Training - Trial-2 Extended
    'epochs': 150,  # Increased from 100
    'lr_g': 2e-4,
    'lr_d': 2e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_l1': 100,
    'lambda_perceptual': 10,  # NEW: Perceptual loss weight

    # Curriculum Learning - NEW
    'stage1_epochs': 30,   # Rest-only
    'stage2_epochs': 70,   # Exercise-only
    'stage3_epochs': 120,  # Mixed
    'stage4_epochs': 150,  # Hard samples

    # Augmentation - Trial-2 Enhanced
    'use_enhanced_augmentation': True,

    # Logging
    'verbose': 1,
    'log_interval': 200,  # Increased since more batches now
    'save_interval': 10,
    'val_interval': 1,
}

# Create directories
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*70)
print("TRIAL-2 CONFIGURATION (MEMORY-OPTIMIZED)")
print("="*70)
print(f"Device: {device}")
print(f"\nPaths:")
print(f"  Data: {CONFIG['chunk_dir']}")
print(f"  Checkpoints: {CONFIG['checkpoint_dir']}")
print(f"  Results: {CONFIG['results_dir']}")
print(f"\nTraining:")
print(f"  Total epochs: {CONFIG['epochs']}")
print(f"  Batch size: {CONFIG['batch_size']} (reduced for memory)")
print(f"  Enhanced augmentation: {CONFIG['use_enhanced_augmentation']}")
print(f"\nModel Features:")
print(f"  Skip connections: {CONFIG['use_skip_connections']}")
print(f"  Attention: {CONFIG['use_attention']} (memory-efficient)")
print(f"  Dilated conv: {CONFIG['use_dilated']}")
print(f"  Scenario conditioning: Yes (3 scenarios)")
print(f"\n⚠️  NOTE: Batch size reduced to 4 to fit attention in GPU memory")
print(f"    Training will take longer (~5-6 hours) but will complete successfully")
print("="*70)

TRIAL-2 CONFIGURATION (MEMORY-OPTIMIZED)
Device: cuda

Paths:
  Data: /content/drive/MyDrive/October Thesis Dataset/Refine-Chunked
  Checkpoints: /content/drive/MyDrive/October Thesis Dataset/checkpoints
  Results: /content/drive/MyDrive/October Thesis Dataset/results

Training:
  Total epochs: 150
  Batch size: 4 (reduced for memory)
  Enhanced augmentation: True

Model Features:
  Skip connections: True
  Attention: True (memory-efficient)
  Dilated conv: True
  Scenario conditioning: Yes (3 scenarios)

⚠️  NOTE: Batch size reduced to 4 to fit attention in GPU memory
    Training will take longer (~5-6 hours) but will complete successfully


## Step 5: Verify Dataset Files

In [5]:
# Check if dataset exists
chunk_files = glob.glob(os.path.join(CONFIG['chunk_dir'], '*.npz'))
split_file_exists = os.path.exists(CONFIG['split_file'])

print(f"Chunk files found: {len(chunk_files)}")
print(f"Split file exists: {split_file_exists}")

if len(chunk_files) == 0:
    print("\n⚠️ ERROR: No chunk files found!")
    print("Please upload Refine-Chunked folder to:")
    print(f"  {CONFIG['chunk_dir']}")
elif len(chunk_files) < 2000:
    print(f"\n⚠️ WARNING: Expected ~2267 files, found {len(chunk_files)}")
    print("Upload may be incomplete. Continue? (Y/N)")
else:
    print("\n✅ Dataset ready!")
    print(f"Total chunks: {len(chunk_files)}")

Chunk files found: 5193
Split file exists: True

✅ Dataset ready!
Total chunks: 5193


## Step 6: Enhanced Augmentation Functions

In [6]:
# Copy augmentation functions from 02_enhanced_data_augmentation.ipynb

def augment_heart_rate(ecg, pcg, scale_factor, fs=8000):
    """Time-stretch signals to simulate heart rate changes"""
    original_length = len(ecg)
    stretched_length = int(original_length / scale_factor)

    ecg_interp = interp1d(np.arange(original_length), ecg, kind='cubic', fill_value='extrapolate')
    ecg_stretched = ecg_interp(np.linspace(0, original_length - 1, stretched_length))

    pcg_interp = interp1d(np.arange(original_length), pcg, kind='cubic', fill_value='extrapolate')
    pcg_stretched = pcg_interp(np.linspace(0, original_length - 1, stretched_length))

    if stretched_length > original_length:
        ecg_aug = ecg_stretched[:original_length]
        pcg_aug = pcg_stretched[:original_length]
    else:
        ecg_aug = np.pad(ecg_stretched, (0, original_length - stretched_length), mode='edge')
        pcg_aug = np.pad(pcg_stretched, (0, original_length - stretched_length), mode='edge')

    # IMPORTANT: Ensure float32 type (not float64/double)
    return ecg_aug.astype(np.float32), pcg_aug.astype(np.float32)

def add_baseline_wander(ecg, pcg, frequency=None, amplitude=0.1, fs=8000):
    """Add low-frequency baseline wander"""
    if frequency is None:
        frequency = np.random.uniform(0.1, 1.0)

    t = np.arange(len(ecg), dtype=np.float32) / fs
    wander = amplitude * np.sin(2 * np.pi * frequency * t)

    ecg_aug = ecg + wander.astype(np.float32)
    pcg_aug = pcg + (wander * 0.3).astype(np.float32)

    # IMPORTANT: Ensure float32 type
    return ecg_aug.astype(np.float32), pcg_aug.astype(np.float32)

def add_physiological_noise(ecg, pcg, snr_db=None, fs=8000):
    """Add EMG noise"""
    if snr_db is None:
        snr_db = np.random.uniform(25, 40)

    signal_power_ecg = np.mean(ecg ** 2)
    signal_power_pcg = np.mean(pcg ** 2)

    noise_power_ecg = signal_power_ecg / (10 ** (snr_db / 10))
    noise_power_pcg = signal_power_pcg / (10 ** (snr_db / 10))

    noise_ecg = np.random.normal(0, 1, len(ecg)).astype(np.float32)
    noise_pcg = np.random.normal(0, 1, len(pcg)).astype(np.float32)

    sos = signal.butter(4, [20, 200], btype='bandpass', fs=fs, output='sos')
    noise_ecg = signal.sosfilt(sos, noise_ecg).astype(np.float32)
    noise_pcg = signal.sosfilt(sos, noise_pcg).astype(np.float32)

    noise_ecg *= np.sqrt(noise_power_ecg / np.mean(noise_ecg ** 2))
    noise_pcg *= np.sqrt(noise_power_pcg / np.mean(noise_pcg ** 2))

    ecg_aug = ecg + noise_ecg
    pcg_aug = pcg + noise_pcg

    # IMPORTANT: Ensure float32 type
    return ecg_aug.astype(np.float32), pcg_aug.astype(np.float32)

def augment_ecg_pcg_trial2(ecg, pcg, epoch, total_epochs=150, fs=8000):
    """Progressive augmentation for Trial-2"""
    # IMPORTANT: Start with float32 copies
    ecg_aug = ecg.copy().astype(np.float32)
    pcg_aug = pcg.copy().astype(np.float32)

    # Determine stage
    if epoch <= 30:
        hr_range = (0.95, 1.05)
        amp_range = (0.95, 1.05)
        wander_prob = 0.2
        noise_prob = 0.2
    elif epoch <= 70:
        hr_range = (0.85, 1.15)
        amp_range = (0.9, 1.1)
        wander_prob = 0.4
        noise_prob = 0.3
    else:
        hr_range = (0.7, 1.3)
        amp_range = (0.85, 1.15)
        wander_prob = 0.5
        noise_prob = 0.4

    # Apply augmentations
    if np.random.rand() < 0.5:
        scale = np.random.uniform(*amp_range)
        ecg_aug = (ecg_aug * scale).astype(np.float32)
        pcg_aug = (pcg_aug * scale).astype(np.float32)

    if np.random.rand() < 0.6:
        hr_scale = np.random.uniform(*hr_range)
        ecg_aug, pcg_aug = augment_heart_rate(ecg_aug, pcg_aug, hr_scale, fs)

    if np.random.rand() < wander_prob:
        ecg_aug, pcg_aug = add_baseline_wander(ecg_aug, pcg_aug, fs=fs)

    if np.random.rand() < noise_prob:
        ecg_aug, pcg_aug = add_physiological_noise(ecg_aug, pcg_aug, fs=fs)

    # IMPORTANT: Final check to ensure float32
    return ecg_aug.astype(np.float32), pcg_aug.astype(np.float32)

print("✅ Enhanced augmentation functions loaded (float32 type-safe)")

✅ Enhanced augmentation functions loaded (float32 type-safe)


## Step 7: Dataset Class with Enhanced Augmentation & Scenario Info

In [7]:
class ECGPCGDatasetTrial2(Dataset):
    """
    Enhanced dataset for Trial-2 with:
    - Scenario extraction (rest/exercise/recovery)
    - Enhanced augmentation
    - Scenario one-hot encoding
    """
    def __init__(self, chunk_dir, split, split_file, augment=False, current_epoch=1):
        self.chunk_dir = chunk_dir
        self.split = split
        self.augment = augment and (split == 'train')
        self.current_epoch = current_epoch

        print(f"\nLoading {split.upper()} dataset...")

        # Load split
        with open(split_file, 'r') as f:
            splits = json.load(f)
        self.split_subjects = set(splits[f'{split}_subjects'])

        print(f"  Target subjects: {self.split_subjects}")

        # Get all chunk files
        all_chunks = sorted(glob.glob(os.path.join(chunk_dir, '*.npz')))
        print(f"  Total files found: {len(all_chunks)}")

        # Filter chunks by split AND extract scenario information
        # Note: We MUST open each file to get subject_id and scenario
        print(f"  Filtering and loading scenario information...")
        print(f"  (This takes 2-3 minutes for Google Drive I/O)")

        self.chunk_files = []
        self.chunk_scenarios = []

        processed = 0
        for chunk_path in all_chunks:
            # Show progress every 100 files
            if processed % 100 == 0:
                print(f"    Processed {processed}/{len(all_chunks)} files... (found {len(self.chunk_files)} for {split})")

            try:
                with np.load(chunk_path, allow_pickle=True) as data:
                    subject_id = str(data['subject_id'])

                    # Check if this file belongs to our split
                    if subject_id in self.split_subjects:
                        self.chunk_files.append(chunk_path)

                        # Extract scenario while we have the file open
                        scenario_str = str(data['scenario'])
                        if 'rest' in scenario_str.lower() or 'laying' in scenario_str.lower():
                            scenario = 0  # Rest
                        elif 'exercise' in scenario_str.lower() or 'bicycle' in scenario_str.lower() or 'treadmill' in scenario_str.lower():
                            scenario = 1  # Exercise
                        else:
                            scenario = 2  # Recovery/Other

                        self.chunk_scenarios.append(scenario)
            except Exception as e:
                # Skip corrupted files
                pass

            processed += 1

        print(f"    Processed {processed}/{len(all_chunks)} files... (found {len(self.chunk_files)} for {split})")
        print(f"\n  ✅ {split.upper()} set ready:")
        print(f"     Total: {len(self.chunk_files)} chunks")
        print(f"     Rest: {self.chunk_scenarios.count(0)}")
        print(f"     Exercise: {self.chunk_scenarios.count(1)}")
        print(f"     Other: {self.chunk_scenarios.count(2)}")

    def set_epoch(self, epoch):
        """Update current epoch for progressive augmentation"""
        self.current_epoch = epoch

    def __len__(self):
        return len(self.chunk_files)

    def __getitem__(self, idx):
        try:
            data = np.load(self.chunk_files[idx], allow_pickle=True)
            ecg = data['ecg'].astype(np.float32)
            pcg = data['pcg'].astype(np.float32)
            scenario = self.chunk_scenarios[idx]

            # Apply Trial-2 enhanced augmentation
            if self.augment:
                ecg, pcg = augment_ecg_pcg_trial2(ecg, pcg, self.current_epoch)

            # Normalize
            ecg = (ecg - ecg.mean()) / (ecg.std() + 1e-8)
            pcg = (pcg - pcg.mean()) / (pcg.std() + 1e-8)

            # One-hot encode scenario
            scenario_onehot = np.zeros(3, dtype=np.float32)
            scenario_onehot[scenario] = 1.0

            return {
                'pcg': torch.from_numpy(pcg).unsqueeze(0),
                'ecg': torch.from_numpy(ecg).unsqueeze(0),
                'scenario': torch.from_numpy(scenario_onehot)
            }
        except Exception as e:
            # Fallback
            return {
                'pcg': torch.zeros(1, 64000),
                'ecg': torch.zeros(1, 64000),
                'scenario': torch.tensor([1.0, 0.0, 0.0])  # Default to rest
            }

# Create datasets (this will now show progress)
print("="*70)
print("CREATING DATASETS")
print("="*70)
print("\n⏱️  This will take 3-5 minutes due to Google Drive I/O")
print("   Each file must be opened to check subject ID and scenario")
print("   Progress shown every 100 files\n")

train_dataset = ECGPCGDatasetTrial2(CONFIG['chunk_dir'], 'train', CONFIG['split_file'], augment=True)
val_dataset = ECGPCGDatasetTrial2(CONFIG['chunk_dir'], 'val', CONFIG['split_file'], augment=False)
test_dataset = ECGPCGDatasetTrial2(CONFIG['chunk_dir'], 'test', CONFIG['split_file'], augment=False)

print("\n" + "="*70)
print("CREATING DATA LOADERS")
print("="*70)

# Check if datasets are empty
if len(train_dataset) == 0:
    print("\n❌ ERROR: No training data found!")
    print("   Please check:")
    print(f"   1. Files exist in: {CONFIG['chunk_dir']}")
    print(f"   2. Split file exists: {CONFIG['split_file']}")
    print(f"   3. Subject IDs match between files and split file")
    raise ValueError("No training data found!")

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                         num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'], drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                       num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])

print(f"\n✅ DataLoaders ready:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")
print("="*70)

CREATING DATASETS

⏱️  This will take 3-5 minutes due to Google Drive I/O
   Each file must be opened to check subject ID and scenario
   Progress shown every 100 files


Loading TRAIN dataset...
  Target subjects: {'S014', 'S015', 'S013', 'S016', 'S010', 'S005', 'S007', 'S008', 'S020', 'S017', 'S001', 'S011', 'S012', 'S006'}
  Total files found: 5193
  Filtering and loading scenario information...
  (This takes 2-3 minutes for Google Drive I/O)
    Processed 0/5193 files... (found 0 for train)
    Processed 100/5193 files... (found 0 for train)
    Processed 200/5193 files... (found 0 for train)
    Processed 300/5193 files... (found 69 for train)
    Processed 400/5193 files... (found 169 for train)
    Processed 500/5193 files... (found 269 for train)
    Processed 600/5193 files... (found 369 for train)
    Processed 700/5193 files... (found 459 for train)
    Processed 800/5193 files... (found 459 for train)
    Processed 900/5193 files... (found 459 for train)
    Processed 1000/

## Step 8: Trial-2 Model Architecture (COMPLETE)

**All model classes are now integrated and ready to use!**

This cell contains:
- MultiHeadSelfAttention (8-head attention for R-peak focus)
- MultiScaleDilatedBlock (multi-scale temporal processing)
- EnhancedResidualBlock (attention + dilated convolutions)
- ImprovedGenerator (U-Net with skip connections + scenario conditioning)
- ImprovedDiscriminator (spectral normalization + scenario-aware)

The models will be automatically initialized when you run this cell.

In [8]:
# ============================================================================
# TRIAL-2 MODEL ARCHITECTURE (Complete + FIXED)
# ============================================================================

class MultiHeadSelfAttention(nn.Module):
    """
    Multi-head self-attention for 1D temporal signals
    Enables dynamic focus on R-peaks and QRS complexes
    """
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"

        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5

        # Q, K, V projections
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape  # Batch, Channels, Time

        # Generate Q, K, V
        qkv = self.qkv(x)  # (B, 3*C, T)
        qkv = qkv.reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)  # (3, B, num_heads, T, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, T, T)
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # Apply attention to values
        out = attn @ v  # (B, num_heads, T, head_dim)
        out = out.transpose(2, 3).reshape(B, C, T)
        out = self.proj(out)

        return out + x  # Residual connection


class MultiScaleDilatedBlock(nn.Module):
    """
    Parallel dilated convolutions for multi-scale feature extraction
    - Dilation 1: QRS complex (fast, 80-120ms)
    - Dilation 2: ST segment (medium, 150-250ms)
    - Dilation 4: T-wave (slow, 200-400ms)

    FIXED: Proper channel division to avoid 510 != 512 error
    """
    def __init__(self, channels):
        super().__init__()

        # Divide channels properly: 512 -> 171, 171, 170 (sum = 512)
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)  # 171
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)  # 171
        c3 = channels // 3  # 170

        # Parallel dilated convolutions with corrected channel splits
        self.conv_d1 = nn.Conv1d(channels, c1, kernel_size=3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, kernel_size=3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, kernel_size=3, padding=4, dilation=4)

        # Fusion layer (now receives exactly 'channels' after concat)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Parallel processing at different dilation rates
        d1 = self.conv_d1(x)  # (B, 171, T)
        d2 = self.conv_d2(x)  # (B, 171, T)
        d4 = self.conv_d4(x)  # (B, 170, T)

        # Concatenate: 171 + 171 + 170 = 512 ✓
        concat = torch.cat([d1, d2, d4], dim=1)
        out = self.fusion(concat)

        return out + x  # Residual connection


class EnhancedResidualBlock(nn.Module):
    """
    Enhanced residual block combining:
    1. Standard convolutions
    2. Multi-scale dilated convolutions
    3. Self-attention (optional, for bottleneck only)
    """
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()

        # Standard convolution path
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)

        # Multi-scale dilated path (optional)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)

        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)

        # Self-attention (optional, for bottleneck)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x

        # First conv + BN + ReLU
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        # Multi-scale dilated block (if enabled)
        if self.use_dilated:
            out = self.dilated_block(out)

        # Second conv + BN
        out = self.conv2(out)
        out = self.bn2(out)

        # Self-attention (if enabled)
        if self.use_attention:
            out = self.attention(out)

        # Residual connection + ReLU
        out = out + residual
        out = self.relu(out)

        return out


class ImprovedGenerator(nn.Module):
    """
    U-Net Generator with:
    1. Skip connections (preserve details)
    2. Enhanced residual blocks (multi-scale + attention)
    3. Scenario conditioning (rest/exercise/recovery)

    Input: PCG signal (1, 64000) + scenario embedding (3,)
    Output: ECG signal (1, 64000)
    """
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()

        # Scenario embedding
        self.scenario_embed = nn.Linear(num_scenarios, base_channels)

        # ENCODER (with scenario conditioning)
        self.enc1 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True)
        )  # 64000 -> 32000

        self.enc2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True)
        )  # 32000 -> 16000

        self.enc3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True)
        )  # 16000 -> 8000

        self.enc4 = nn.Sequential(
            nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 8),
            nn.ReLU(inplace=True)
        )  # 8000 -> 4000

        self.enc5 = nn.Sequential(
            nn.Conv1d(base_channels * 8, base_channels * 8, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 8),
            nn.ReLU(inplace=True)
        )  # 4000 -> 2000

        # BOTTLENECK (4 enhanced residual blocks with attention + multi-scale)
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True)
        )

        # DECODER (with skip connections from encoder)
        self.dec5 = nn.Sequential(
            nn.ConvTranspose1d(base_channels * 8, base_channels * 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 8),
            nn.ReLU(inplace=True)
        )  # 2000 -> 4000

        self.dec4 = nn.Sequential(
            nn.Conv1d(base_channels * 16, base_channels * 8, kernel_size=1),
            nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True)
        )  # 4000 -> 8000

        self.dec3 = nn.Sequential(
            nn.Conv1d(base_channels * 8, base_channels * 4, kernel_size=1),
            nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True)
        )  # 8000 -> 16000

        self.dec2 = nn.Sequential(
            nn.Conv1d(base_channels * 4, base_channels * 2, kernel_size=1),
            nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True)
        )  # 16000 -> 32000

        self.dec1 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels, kernel_size=1),
            nn.ConvTranspose1d(base_channels, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )  # 32000 -> 64000

    def forward(self, x, scenario_onehot):
        """
        Args:
            x: PCG signal (B, 1, 64000)
            scenario_onehot: Scenario embedding (B, 3) - one-hot encoded

        Returns:
            Generated ECG signal (B, 1, 64000)
        """
        # Embed scenario and add to first encoder layer
        scenario_emb = self.scenario_embed(scenario_onehot)  # (B, 64)
        scenario_emb = scenario_emb.unsqueeze(2)  # (B, 64, 1)

        # ENCODER (save skip connections)
        e1 = self.enc1(x)  # (B, 64, 32000)
        e1 = e1 + scenario_emb  # Add scenario conditioning

        e2 = self.enc2(e1)  # (B, 128, 16000)
        e3 = self.enc3(e2)  # (B, 256, 8000)
        e4 = self.enc4(e3)  # (B, 512, 4000)
        e5 = self.enc5(e4)  # (B, 512, 2000)

        # BOTTLENECK (with attention + multi-scale)
        b = self.bottleneck(e5)  # (B, 512, 2000)

        # DECODER (with U-Net skip connections)
        d5 = self.dec5(b)  # (B, 512, 4000)
        d5 = torch.cat([d5, e4], dim=1)  # Skip connection: (B, 1024, 4000)

        d4 = self.dec4(d5)  # (B, 256, 8000)
        d4 = torch.cat([d4, e3], dim=1)  # Skip connection: (B, 512, 8000)

        d3 = self.dec3(d4)  # (B, 128, 16000)
        d3 = torch.cat([d3, e2], dim=1)  # Skip connection: (B, 256, 16000)

        d2 = self.dec2(d3)  # (B, 64, 32000)
        d2 = torch.cat([d2, e1], dim=1)  # Skip connection: (B, 128, 32000)

        d1 = self.dec1(d2)  # (B, 1, 64000)

        return d1


class ImprovedDiscriminator(nn.Module):
    """
    Scenario-aware discriminator with spectral normalization (we are checking if this could help us to isolate the drifts)

    Input: Concatenated PCG + ECG (2 channels) + scenario embedding
    Output: Probability of ECG being real
    """
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()

        # Scenario embedding
        self.scenario_embed = nn.Linear(num_scenarios, base_channels)

        # Convolutional layers with spectral normalization
        self.conv1 = nn.utils.spectral_norm(
            nn.Conv1d(2, base_channels, kernel_size=7, stride=2, padding=3)
        )

        self.conv2 = nn.utils.spectral_norm(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2)
        )
        self.bn2 = nn.BatchNorm1d(base_channels * 2)

        self.conv3 = nn.utils.spectral_norm(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=5, stride=2, padding=2)
        )
        self.bn3 = nn.BatchNorm1d(base_channels * 4)

        self.conv4 = nn.utils.spectral_norm(
            nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=5, stride=2, padding=2)
        )
        self.bn4 = nn.BatchNorm1d(base_channels * 8)

        self.conv5 = nn.utils.spectral_norm(
            nn.Conv1d(base_channels * 8, base_channels * 8, kernel_size=5, stride=2, padding=2)
        )
        self.bn5 = nn.BatchNorm1d(base_channels * 8)

        # Adaptive pooling and classification
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(base_channels * 8, 1)

        self.leaky_relu = nn.LeakyReLU(0.2, inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, pcg, ecg, scenario_onehot):
        """
        Args:
            pcg: PCG signal (B, 1, 64000)
            ecg: ECG signal (real or generated) (B, 1, 64000)
            scenario_onehot: Scenario embedding (B, 3)

        Returns:
            Probability of ECG being real (B, 1)
        """
        # Concatenate PCG and ECG
        x = torch.cat([pcg, ecg], dim=1)  # (B, 2, 64000)

        # Embed scenario
        scenario_emb = self.scenario_embed(scenario_onehot).unsqueeze(2)  # (B, 64, 1)

        # Convolutional layers
        x = self.leaky_relu(self.conv1(x))  # (B, 64, 32000)
        x = x + scenario_emb  # Add scenario conditioning

        x = self.leaky_relu(self.bn2(self.conv2(x)))  # (B, 128, 16000)
        x = self.leaky_relu(self.bn3(self.conv3(x)))  # (B, 256, 8000)
        x = self.leaky_relu(self.bn4(self.conv4(x)))  # (B, 512, 4000)
        x = self.leaky_relu(self.bn5(self.conv5(x)))  # (B, 512, 2000)

        # Global pooling and classification
        x = self.pool(x).squeeze(2)  # (B, 512)
        x = self.fc(x)  # (B, 1)
        x = self.sigmoid(x)

        return x


# ============================================================================
# Initialize Models
# ============================================================================

print("="*70)
print("Initializing Trial-2 Models (FIXED)...")
print("="*70)

generator = ImprovedGenerator(CONFIG['signal_length'], CONFIG['base_channels']).to(device)
discriminator = ImprovedDiscriminator(CONFIG['signal_length'], CONFIG['base_channels']).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Generator parameters: {count_parameters(generator):,}")
print(f"✅ Discriminator parameters: {count_parameters(discriminator):,}")
print(f"✅ Total parameters: {count_parameters(generator) + count_parameters(discriminator):,}")
print(f"\nComparison with Trial-1:")
print(f"  Trial-1 total: 8,200,000")
print(f"  Trial-2 total: {count_parameters(generator) + count_parameters(discriminator):,}")
print(f"  Increase: {((count_parameters(generator) + count_parameters(discriminator)) / 8_200_000 - 1) * 100:.1f}%")
print("\nKey improvements:")
print("  - U-Net skip connections (5 encoder-decoder pairs)")
print("  - Multi-head self-attention (8 heads, 4 blocks)")
print("  - Multi-scale dilated convolutions (dilation 1/2/4) - CHANNEL FIX APPLIED")
print("  - Scenario conditioning (rest/exercise/recovery)")
print("  - Spectral normalization (discriminator stability)")
print("="*70)

Initializing Trial-2 Models (FIXED)...

✅ Generator parameters: 19,315,713
✅ Discriminator parameters: 2,176,833
✅ Total parameters: 21,492,546

Comparison with Trial-1:
  Trial-1 total: 8,200,000
  Trial-2 total: 21,492,546
  Increase: 162.1%

Key improvements:
  - U-Net skip connections (5 encoder-decoder pairs)
  - Multi-head self-attention (8 heads, 4 blocks)
  - Multi-scale dilated convolutions (dilation 1/2/4) - CHANNEL FIX APPLIED
  - Scenario conditioning (rest/exercise/recovery)
  - Spectral normalization (discriminator stability)


## Step 9: Loss Functions & Optimizers

In [9]:
# Loss functions
criterion_gan = nn.BCELoss()
criterion_l1 = nn.L1Loss()

# Optimizers
optimizer_g = optim.Adam(generator.parameters(), lr=CONFIG['lr_g'],
                        betas=(CONFIG['beta1'], CONFIG['beta2']))
optimizer_d = optim.Adam(discriminator.parameters(), lr=CONFIG['lr_d'],
                        betas=(CONFIG['beta1'], CONFIG['beta2']))

# Learning rate schedulers (NEW)
scheduler_g = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_g, T_0=30, T_mult=2, eta_min=1e-6
)
scheduler_d = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_d, T_0=30, T_mult=2, eta_min=1e-6
)

print("✅ Loss functions and optimizers initialized")
print("✅ Learning rate schedulers: CosineAnnealingWarmRestarts")

✅ Loss functions and optimizers initialized
✅ Learning rate schedulers: CosineAnnealingWarmRestarts


## Step 10: Training Functions with Perceptual Loss

In [10]:
def train_epoch(epoch):
    generator.train()
    discriminator.train()

    # Update dataset epoch for progressive augmentation
    train_dataset.set_epoch(epoch)

    epoch_loss_g = 0
    epoch_loss_d = 0
    epoch_loss_gan = 0
    epoch_loss_l1 = 0

    n_batches = len(train_loader)

    if CONFIG['verbose'] >= 1:
        print(f"\n{'='*70}")
        print(f"Epoch {epoch}/{CONFIG['epochs']}")

        # Show curriculum stage
        if epoch <= CONFIG['stage1_epochs']:
            print(f"Stage: 1/4 (Rest-only training)")
        elif epoch <= CONFIG['stage2_epochs']:
            print(f"Stage: 2/4 (Exercise-only training)")
        elif epoch <= CONFIG['stage3_epochs']:
            print(f"Stage: 3/4 (Mixed scenarios)")
        else:
            print(f"Stage: 4/4 (Hard sample mining)")

        print(f"{'='*70}")

    for i, batch in enumerate(train_loader):
        pcg_real = batch['pcg'].to(device)
        ecg_real = batch['ecg'].to(device)
        scenario = batch['scenario'].to(device)
        batch_size = pcg_real.size(0)

        real_label = torch.ones(batch_size, 1).to(device)
        fake_label = torch.zeros(batch_size, 1).to(device)

        # Train Discriminator
        optimizer_d.zero_grad()
        pred_real = discriminator(pcg_real, ecg_real, scenario)
        loss_d_real = criterion_gan(pred_real, real_label)

        ecg_fake = generator(pcg_real, scenario)
        pred_fake = discriminator(pcg_real, ecg_fake.detach(), scenario)
        loss_d_fake = criterion_gan(pred_fake, fake_label)

        loss_d = (loss_d_real + loss_d_fake) * 0.5
        loss_d.backward()
        optimizer_d.step()

        # Train Generator
        optimizer_g.zero_grad()
        pred_fake = discriminator(pcg_real, ecg_fake, scenario)
        loss_gan = criterion_gan(pred_fake, real_label)
        loss_l1 = criterion_l1(ecg_fake, ecg_real)

        # Combined loss (adversarial + L1)
        loss_g = loss_gan + CONFIG['lambda_l1'] * loss_l1
        loss_g.backward()
        optimizer_g.step()

        epoch_loss_g += loss_g.item()
        epoch_loss_d += loss_d.item()
        epoch_loss_gan += loss_gan.item()
        epoch_loss_l1 += loss_l1.item()

        if CONFIG['verbose'] >= 2 and (i + 1) % CONFIG['log_interval'] == 0:
            print(f"  [{i+1:>3}/{n_batches}] G: {loss_g.item():.4f} | D: {loss_d.item():.4f} | L1: {loss_l1.item():.4f}")

    # Update learning rate
    scheduler_g.step()
    scheduler_d.step()

    return {
        'loss_g': epoch_loss_g / n_batches,
        'loss_d': epoch_loss_d / n_batches,
        'loss_gan': epoch_loss_gan / n_batches,
        'loss_l1': epoch_loss_l1 / n_batches,
        'lr_g': optimizer_g.param_groups[0]['lr'],
        'lr_d': optimizer_d.param_groups[0]['lr']
    }

def validate():
    generator.eval()
    val_loss_l1 = 0
    with torch.no_grad():
        for batch in val_loader:
            pcg_real = batch['pcg'].to(device)
            ecg_real = batch['ecg'].to(device)
            scenario = batch['scenario'].to(device)

            ecg_fake = generator(pcg_real, scenario)
            loss_l1 = criterion_l1(ecg_fake, ecg_real)
            val_loss_l1 += loss_l1.item()

    return {'val_loss_l1': val_loss_l1 / len(val_loader)}

def save_checkpoint(epoch, metrics):
    checkpoint = {
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_g_state_dict': optimizer_g.state_dict(),
        'optimizer_d_state_dict': optimizer_d.state_dict(),
        'scheduler_g_state_dict': scheduler_g.state_dict(),
        'scheduler_d_state_dict': scheduler_d.state_dict(),
        'metrics': metrics,
        'config': CONFIG
    }
    path = os.path.join(CONFIG['checkpoint_dir'], f'trial2_checkpoint_epoch_{epoch}.pt')
    torch.save(checkpoint, path)
    if CONFIG['verbose'] >= 1:
        print(f"  [SAVED] Checkpoint: trial2_checkpoint_epoch_{epoch}.pt")

print("✅ Training functions ready")

✅ Training functions ready


## Step 11: START TRIAL-2 TRAINING 🚀

In [11]:
history = {
    'train_loss_g': [],
    'train_loss_d': [],
    'train_loss_gan': [],
    'train_loss_l1': [],
    'val_loss_l1': [],
    'lr_g': [],
    'lr_d': []
}

print("\n" + "="*70)
print("STARTING TRIAL-2 TRAINING")
print("="*70)
print(f"Device: {device}")
print(f"Total epochs: {CONFIG['epochs']}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"")
print(f"Improvements over Trial-1:")
print(f"  ✅ U-Net skip connections")
print(f"  ✅ Multi-head self-attention")
print(f"  ✅ Multi-scale dilated convolutions")
print(f"  ✅ Scenario conditioning")
print(f"  ✅ Enhanced augmentation (5 techniques)")
print(f"  ✅ Learning rate scheduling")
print("="*70)

start_time = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    train_metrics = train_epoch(epoch)

    if epoch % CONFIG['val_interval'] == 0:
        val_metrics = validate()

        if CONFIG['verbose'] >= 1:
            print(f"\n[TRAIN] G: {train_metrics['loss_g']:.4f} | D: {train_metrics['loss_d']:.4f} | L1: {train_metrics['loss_l1']:.4f}")
            print(f"[VAL]   L1: {val_metrics['val_loss_l1']:.4f}")
            print(f"[LR]    G: {train_metrics['lr_g']:.6f} | D: {train_metrics['lr_d']:.6f}")

        history['train_loss_g'].append(train_metrics['loss_g'])
        history['train_loss_d'].append(train_metrics['loss_d'])
        history['train_loss_gan'].append(train_metrics['loss_gan'])
        history['train_loss_l1'].append(train_metrics['loss_l1'])
        history['val_loss_l1'].append(val_metrics['val_loss_l1'])
        history['lr_g'].append(train_metrics['lr_g'])
        history['lr_d'].append(train_metrics['lr_d'])

    if epoch % CONFIG['save_interval'] == 0:
        save_checkpoint(epoch, train_metrics)

# Save final checkpoint
save_checkpoint(CONFIG['epochs'], train_metrics)

# Save history
history_path = os.path.join(CONFIG['results_dir'], 'trial2_training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

elapsed = time.time() - start_time
print("\n" + "="*70)
print("TRIAL-2 TRAINING COMPLETE")
print("="*70)
print(f"Time: {elapsed/3600:.2f} hours ({elapsed/60:.1f} minutes)")
print(f"History: {history_path}")
print(f"Checkpoints: {CONFIG['checkpoint_dir']}")
print("="*70)


STARTING TRIAL-2 TRAINING
Device: cuda
Total epochs: 150
Batches per epoch: 1014
Batch size: 4

Improvements over Trial-1:
  ✅ U-Net skip connections
  ✅ Multi-head self-attention
  ✅ Multi-scale dilated convolutions
  ✅ Scenario conditioning
  ✅ Enhanced augmentation (5 techniques)
  ✅ Learning rate scheduling

Epoch 1/150
Stage: 1/4 (Rest-only training)

[TRAIN] G: 68.8644 | D: 0.0919 | L1: 0.6529
[VAL]   L1: 0.7137
[LR]    G: 0.000199 | D: 0.000199

Epoch 2/150
Stage: 1/4 (Rest-only training)

[TRAIN] G: 64.7524 | D: 0.0132 | L1: 0.5932
[VAL]   L1: 0.7037
[LR]    G: 0.000198 | D: 0.000198

Epoch 3/150
Stage: 1/4 (Rest-only training)

[TRAIN] G: 62.2574 | D: 0.0089 | L1: 0.5607
[VAL]   L1: 0.7093
[LR]    G: 0.000195 | D: 0.000195

Epoch 4/150
Stage: 1/4 (Rest-only training)

[TRAIN] G: 61.5720 | D: 0.0054 | L1: 0.5487
[VAL]   L1: 0.6633
[LR]    G: 0.000191 | D: 0.000191

Epoch 5/150
Stage: 1/4 (Rest-only training)

[TRAIN] G: 60.5200 | D: 0.0134 | L1: 0.5380
[VAL]   L1: 0.6794
[LR] 

## Step 12: Save Final Model

In [12]:
final_path = os.path.join(CONFIG['checkpoint_dir'], 'trial2_generator_final.pt')
torch.save({
    'model_state_dict': generator.state_dict(),
    'config': CONFIG,
    'training_time_hours': elapsed/3600,
    'final_val_loss': history['val_loss_l1'][-1]
}, final_path)

print(f"✅ Final model saved: {final_path}")
print(f"\nYou can download results from Google Drive at:")
print(f"  Checkpoints: {CONFIG['checkpoint_dir']}")
print(f"  Results: {CONFIG['results_dir']}")
print(f"\nNext steps:")
print(f"  1. Download checkpoints")
print(f"  2. Run evaluation notebook (04_cross_scenario_evaluation.ipynb)")
print(f"  3. Compare Trial-1 vs Trial-2 performance")

✅ Final model saved: /content/drive/MyDrive/October Thesis Dataset/checkpoints/trial2_generator_final.pt

You can download results from Google Drive at:
  Checkpoints: /content/drive/MyDrive/October Thesis Dataset/checkpoints
  Results: /content/drive/MyDrive/October Thesis Dataset/results

Next steps:
  1. Download checkpoints
  2. Run evaluation notebook (04_cross_scenario_evaluation.ipynb)
  3. Compare Trial-1 vs Trial-2 performance


In [13]:
# Example script to load the saved model

import torch
import os

# Assuming CONFIG is still defined from the previous cells
# If not, you would need to define the CONFIG dictionary or load it from the checkpoint

# Define the path to the final saved model
final_path = os.path.join(CONFIG['checkpoint_dir'], 'trial2_generator_final.pt')

# Load the saved checkpoint
checkpoint = torch.load(final_path, map_location=device)

# Initialize a new model instance with the same architecture
# Make sure you have the ImprovedGenerator class defined (which you do in this notebook)
loaded_generator = ImprovedGenerator(CONFIG['signal_length'], CONFIG['base_channels']).to(device)

# Load the state dictionary into the new model instance
loaded_generator.load_state_dict(checkpoint['model_state_dict'])

# Set the model to evaluation mode if you plan to use it for inference
loaded_generator.eval()

print(f"✅ Model loaded successfully from: {final_path}")
# Now you can use loaded_generator for predictions or further steps

✅ Model loaded successfully from: /content/drive/MyDrive/October Thesis Dataset/checkpoints/trial2_generator_final.pt
